# Stage 8 — Official StableMate LOCO (D3 held out)

Runs the **original author's StableMate R implementation** at pinned commit `cf0fe9f344c1756b6f89fd33f9fdae1a7f35b24a` for CD versus Not IBD. The three training cohorts are the model's environments; **the held-out cohort never enters gene selection or model fitting**.

- Target: comparative study on our 4-cohort data, **not** reproduction of StableMate's published results.
- Logistic base regressor; original author prefilter `lasso_prefilt(p=50,K=2000,family='binomial')`; ST2* logistic predictive/stability objectives; native author ensemble prediction.
- Main ensemble **K=2000** matches the paper's reported case-study count, and random-Lasso prefilter **K=2000** supplies a per-iteration candidate pool. The dataset, candidate count p=50 and LOCO evaluation remain adaptations; this is not reproduction of their original published numerical results.
- A native StableMate predictor is **not the same classifier** as our LASSO/Elastic Net model, so comparisons are observational, not strict selector-only ablations.
- Phases save to Drive after author prefilter and after official model fitting. Re-running this notebook resumes saved phases if the runtime disconnects; **a phase interrupted mid-computation must restart; this could cost hours at K=2000**.
- Never call this a single-cohort independent-signature experiment: StableMate stability needs multiple environments.

**Google Colab CPU.** Output: `MyDrive/CMPS355_Research/runs/stablemate_k2000/d3/`. No other project folders are altered.

In [8]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4399.99
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

In [1]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, tarfile, tempfile, time
from datetime import datetime, timezone
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CMPS355_Research')
DATA = ROOT / 'data/validated/ibd_dataset_sample_aligned.zip'
assert ROOT.is_dir(), f'Project not found: {ROOT}'
assert DATA.is_file(), f'Accepted dataset not found: {DATA}'
HELDOUT = 'd3'
OUT = ROOT / 'runs/stablemate_k2000' / HELDOUT
OUT.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/cmps355_stage8') / HELDOUT
WORK.mkdir(parents=True, exist_ok=True)
SRC = Path('/content/cmps355_stage8/src')
SRC.mkdir(parents=True,exist_ok=True)
LIB = Path('/content/cmps355_r_lib')
LIB.mkdir(parents=True,exist_ok=True)
SHA='cf0fe9f344c1756b6f89fd33f9fdae1a7f35b24a'
K = 2000    # Published StableMate case-study ensemble iteration count.
PREFILTER_P = 50
BOOTSTRAP_B = 1000
NCORE = 2
print('Project:', ROOT)
print('Output:', OUT)
print('Held-out cohort:', HELDOUT)


Mounted at /content/drive
Project: /content/drive/MyDrive/CMPS355_Research
Output: /content/drive/MyDrive/CMPS355_Research/runs/stablemate_k2000/d3
Held-out cohort: d3


## Official source files

In [2]:
%%writefile /content/cmps355_stage8/src/full_gene_loco.py
"""Independent-cohort full-gene LASSO / Elastic Net nested evaluation."""
from __future__ import annotations

import csv
import hashlib
import io
import json
import os
import platform
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

EXPECTED_ZIP_SHA = '8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74'
EXPECTED_CSV_SHA = {
    'count_d1_aligned.csv': '0f595746ea572a349580d9d3c0242559345b88c927c161b1982933102e6c812d',
    'count_d2_aligned.csv': '2ff413b813e3e4d10927da7f8342eccb35a00205c319c0eb714388aea97f245d',
    'count_d3_aligned.csv': '33b2e51e682da4c41f53becf00b1fc88bb5710293ce2d9d87837bca86a7b3d48',
    'count_d4_aligned.csv': 'c26d8db65b335c89967bd4e072c61a6b7adcb57715c5fe36107d9bd8cd9409e0',
    'desc_d1.csv': '91aadec65bb7f2c1945e7941222b9f490a0c9041d495b8d22606f614484c7b33',
    'desc_d2.csv': '4198f574167754eb00e007ded8cfc2528d50ead0d80b96c1a5cc8139131a140b',
    'desc_d3.csv': '9db3c39975a50afcbd851d3cdcdf56f1e59e7f9bbe18b3b80fd7328400a31332',
    'desc_d4.csv': 'bdd5f6d5ae9a6713470f1eadab2e62a961a38d417f27401359d55b5d906a1193',
}
COHORTS = ('d1','d2','d3','d4')
C_GRID = (0.001, 0.01, 0.1, 1.0)
ELASTIC_RATIOS = (0.5, 0.9)
MIN_CPM = 1.0
MIN_FRACTION = 0.1
SEED = 355
MAX_ITER = 1600
TOL = 0.001


def sha_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def require(condition: bool, reason: str) -> None:
    if not condition:
        raise ValueError(reason)


def find_input(directory: Path):
    root = Path(directory)
    require(root.exists(), f'Input root not found: {root}')
    unpacked = [p for p in root.rglob('count_d1_aligned.csv') if p.is_file()]
    for file in unpacked:
        folder = file.parent
        if all((folder / name).exists() for name in EXPECTED_CSV_SHA):
            return ('directory', folder)
    archives = [p for p in root.rglob('ibd_dataset_sample_aligned.zip') if p.is_file()]
    require(len(archives) == 1, 'Attach exactly one accepted dataset input containing the eight aligned CSV files')
    return ('archive', archives[0])


def load_data(directory: Path) -> tuple[pd.DataFrame, np.ndarray, tuple[str,...], dict]:
    mode, source = find_input(directory)
    if mode == 'archive':
        require(sha_bytes(source.read_bytes()) == EXPECTED_ZIP_SHA, 'Dataset ZIP checksum mismatch')
        with ZipFile(source) as zf:
            require(zf.testzip() is None, 'ZIP CRC verification failed')
            raw = {name: zf.read(name) for name in EXPECTED_CSV_SHA}
    else:
        raw = {name: (source / name).read_bytes() for name in EXPECTED_CSV_SHA}
    for name, expected in EXPECTED_CSV_SHA.items():
        require(sha_bytes(raw[name]) == expected, f'Dataset file checksum mismatch: {name}')
    headers = {}
    for cohort in COHORTS:
        names = next(csv.reader(io.StringIO(raw[f'count_{cohort}_aligned.csv'].decode('utf-8-sig'))))
        require(len(names) == len(set(names)) and names[0] == 'ID', f'Invalid gene names: {cohort}')
        headers[cohort] = names
    genes = tuple(sorted(set.intersection(*(set(headers[c][1:]) for c in COHORTS))))
    require(len(genes) == 9983, 'Unexpected shared gene count')
    counts = []
    metadata = []
    expected_rows = dict(zip(COHORTS, (260,179,76,174)))
    for cohort in COHORTS:
        desc = pd.read_csv(io.BytesIO(raw[f'desc_{cohort}.csv']), dtype={'ID':'string','Study':'string','class':'string'})
        df = pd.read_csv(io.BytesIO(raw[f'count_{cohort}_aligned.csv']), usecols=['ID',*genes], dtype={'ID':'string'})
        require(len(desc)==len(df)==expected_rows[cohort], f'{cohort}: incorrect sample count')
        require(desc['ID'].tolist()==df['ID'].tolist(), f'{cohort}: sample alignment mismatch')
        require({'ID','Study','class'} <= set(desc.columns), f'{cohort}: missing clinical fields')
        require(set(desc['class'])=={'CD','Not IBD'}, f'{cohort}: unexpected classes')
        mat = df.loc[:, list(genes)].to_numpy(dtype=np.float64, copy=True)
        require(np.isfinite(mat).all() and (mat>=0).all() and (mat.sum(axis=1)>0).all(), f'{cohort}: invalid counts')
        desc = desc[['ID','Study','class']].copy()
        desc.insert(0,'cohort',cohort)
        metadata.append(desc)
        counts.append(mat)
    meta = pd.concat(metadata, ignore_index=True)
    require(meta['ID'].is_unique and len(meta)==689, 'Duplicate or missing patients')
    require(meta['Study'].nunique()==6, 'BioProject count mismatch')
    return meta, np.vstack(counts), genes, {'source_mode':mode,'source_path':str(source), 'zip_sha256':EXPECTED_ZIP_SHA,'csv_sha256':EXPECTED_CSV_SHA}


def log_cpm(values: np.ndarray) -> np.ndarray:
    depths = values.sum(axis=1)
    require(np.all(depths>0), 'Empty library detected')
    return np.log2(1.0 + 1e6 * values / depths[:, None])


def fit_features(train_counts: np.ndarray) -> tuple[np.ndarray, np.ndarray, StandardScaler, dict]:
    x = log_cpm(train_counts)
    threshold = max(2, int(np.ceil(MIN_FRACTION * len(x))))
    valid = ((x > np.log2(1+MIN_CPM)).sum(axis=0) >= threshold) & (x.var(axis=0) > 1e-12)
    indices = np.flatnonzero(valid)
    require(len(indices) >= 2, 'Insufficient genes after training-only filtering')
    scaler = StandardScaler()
    fitted = scaler.fit_transform(x[:, indices])
    require(np.isfinite(fitted).all(), 'Invalid prepared features')
    return fitted, indices, scaler, {'fit_patients':len(x), 'prevalence_threshold_samples':threshold,'eligible_genes':len(indices)}


def apply_features(counts: np.ndarray, indices: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    return scaler.transform(log_cpm(counts)[:, indices])


def model_fit(x: np.ndarray, y: np.ndarray, C: float, ratio: float):
    sklearn_major_minor = tuple(int(z) for z in sklearn.__version__.split('.')[:2])
    args = {'l1_ratio':float(ratio)} if sklearn_major_minor >= (1,8) else {
        'penalty':'l1' if ratio==1 else 'elasticnet',
        'l1_ratio': None if ratio==1 else float(ratio)}
    model = LogisticRegression(solver='saga',C=float(C),class_weight='balanced',
                               max_iter=MAX_ITER,tol=TOL,random_state=SEED,**args)
    with warnings.catch_warnings(record=True) as recorded:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(x,y)
    converged = not any(issubclass(r.category,ConvergenceWarning) for r in recorded)
    return model, converged


def metrics(y: np.ndarray, prob: np.ndarray) -> dict:
    require(np.isfinite(prob).all() and ((0<=prob)&(prob<=1)).all(), 'Invalid probabilities')
    pred = (prob >= .5).astype(int)
    tn, fp, fn, tp = [int(v) for v in confusion_matrix(y,pred,labels=[0,1]).ravel()]
    return dict(auroc=float(roc_auc_score(y,prob)), average_precision=float(average_precision_score(y,prob)),
                balanced_accuracy=float(balanced_accuracy_score(y,pred)),
                sensitivity=tp/(tp+fn), specificity=tn/(tn+fp),
                tp=tp,tn=tn,fp=fp,fn=fn)


def safe_csv(df: pd.DataFrame, path: Path):
    temporary = path.with_suffix(path.suffix+'.tmp')
    df.to_csv(temporary, index=False)
    os.replace(temporary,path)


def safe_json(obj: dict, path: Path):
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(obj,indent=2,sort_keys=True),encoding='utf-8')
    os.replace(temporary,path)


def run(heldout: str, input_root: Path, output_root: Path) -> Path:
    require(heldout in COHORTS, 'Unknown outer cohort')
    started = time.monotonic()
    meta,counts,genes,source=load_data(input_root)
    y = (meta['class']=='CD').to_numpy(dtype=int)
    cohorts=meta['cohort'].to_numpy()
    test_ids=np.flatnonzero(cohorts==heldout)
    train_ids=np.flatnonzero(cohorts!=heldout)
    require(len(test_ids)>0 and len(train_ids)+len(test_ids)==689, 'Outer fold coverage invalid')
    target=Path(output_root) / heldout
    require(not (target/'report.json').exists(), 'Existing completed results cannot be overwritten')
    target.mkdir(parents=True,exist_ok=True)
    scores=[]
    selected=[]
    predictions=[]
    methods={ 'lasso':(1.0,), 'elastic_net':ELASTIC_RATIOS }
    output={ 'status':'RUNNING', 'heldout':heldout,'dataset':source, 'config':{
        'folds':'nested leave-one-analytical-cohort-out', 'gene_universe':'9983 genes shared before inspecting labels',
        'normalization':'log2(1 + CPM) per sample on shared gene universe',
        'filter':'training-only CPM > 1 in >=10% of training samples, nonzero variance',
        'scaling':'training-only standardization','outer_score_threshold':0.5,
        'tuning_metric':'mean of 3 independent cohort-validation AUROCs',
        'C_grid':C_GRID, 'elastic_ratios':ELASTIC_RATIOS,'max_iter':MAX_ITER,'tol':TOL,
        'class_weight':'balanced','random_seed':SEED}, 'software':{
        'python':platform.python_version(), 'numpy':np.__version__,'pandas':pd.__version__,
        'scipy':scipy.__version__,'sklearn':sklearn.__version__},
        'candidates':[], 'results':[], 'start_utc':datetime.now(timezone.utc).isoformat() }
    safe_json(output,target/'progress.json')
    for method, ratios in methods.items():
        configs=[(C,r) for r in ratios for C in C_GRID]
        candidate_scores={ (float(c),float(r)):[] for c,r in configs }
        for inner in COHORTS:
            if inner == heldout:
                continue
            ix=np.flatnonzero((cohorts!=heldout)&(cohorts!=inner))
            vx=np.flatnonzero(cohorts==inner)
            require(np.intersect1d(np.r_[ix,vx],test_ids).size==0, 'Outer test leakage')
            require(np.intersect1d(ix,vx).size==0, 'Inner sample overlap')
            train_matrix, inds, scaler, prep=fit_features(counts[ix])
            val_matrix=apply_features(counts[vx],inds,scaler)
            print(f'{heldout}/{method}: validating on {inner} using {len(inds)} genes', flush=True)
            for c,r in configs:
                model, conv = model_fit(train_matrix,y[ix],c,r)
                prob=model.predict_proba(val_matrix)[:,1]
                score=metrics(y[vx],prob)
                scores.append(dict(outer=heldout,method=method,validation=inner,C=c,l1_ratio=r,
                                   n_fit=len(ix),n_validation=len(vx),
                                   n_candidate_genes=len(inds), nonzero=int((np.abs(model.coef_[0])>1e-8).sum()),
                                   converged=conv, n_iter=int(model.n_iter_[0]),**score))
                candidate_scores[(float(c),float(r))].append(score['auroc'])
        require(all(len(z)==3 for z in candidate_scores.values()), 'Incomplete inner evaluation')
        ranked=sorted(candidate_scores,key=lambda cr:(-np.mean(candidate_scores[cr]),cr[0],-cr[1]))
        c,r=ranked[0]
        all_x, inds,scaler,preprocess=fit_features(counts[train_ids])
        heldout_x=apply_features(counts[test_ids],inds,scaler)
        final,conv=model_fit(all_x,y[train_ids],c,r)
        pr=final.predict_proba(heldout_x)[:,1]
        result=dict(method=method,outer=heldout,C=c,l1_ratio=r,
                    inner_mean_auroc=float(np.mean(candidate_scores[(c,r)])),
                    train_patients=len(train_ids),test_patients=len(test_ids),
                    candidate_gene_count=len(inds),nonzero_coefficients=int((np.abs(final.coef_[0])>1e-8).sum()),
                    converged=conv,n_iter=int(final.n_iter_[0]),preprocessing=preprocess,**metrics(y[test_ids],pr))
        output['results'].append(result)
        for i,prob in zip(test_ids,pr):
            predictions.append(dict(outer=heldout,method=method,ID=str(meta.iloc[i]['ID']),
                                    Study=str(meta.iloc[i]['Study']),class_true=str(meta.iloc[i]['class']),
                                    y_true=int(y[i]),probability_CD=float(prob)))
        for position in np.flatnonzero(np.abs(final.coef_[0])>1e-8):
            selected.append(dict(outer=heldout,method=method,gene=genes[inds[position]],
                                 coef_scaled=float(final.coef_[0,position])))
        output['candidates'].extend([dict(method=method,C=z[0],l1_ratio=z[1],
                               mean_validation_auroc=float(np.mean(candidate_scores[z]))) for z in ranked])
        safe_csv(pd.DataFrame(scores),target/'inner_scores.csv')
        safe_csv(pd.DataFrame(predictions),target/'predictions.csv')
        safe_csv(pd.DataFrame(selected,columns=['outer','method','gene','coef_scaled']),target/'selected_genes.csv')
        safe_json(output,target/'progress.json')
        print(f"{heldout}/{method}: AUROC={result['auroc']:.3f}, selected={result['nonzero_coefficients']}, converged={conv}",flush=True)
    require(len(output['results'])==2 and len(predictions)==2*len(test_ids),'Incomplete outer fold')
    require(all(len({x['ID'] for x in predictions if x['method']==m})==len(test_ids) for m in methods),
            'Duplicate outer predictions')
    output['status']='PASS' if all(item['converged'] for item in output['results']) and all(s['converged'] for s in scores) else 'WARN_CONVERGENCE'
    output['elapsed_seconds']=round(time.monotonic()-started,2)
    output['completed_utc']=datetime.now(timezone.utc).isoformat()
    output['artifacts_sha256']={f:hashlib.sha256((target/f).read_bytes()).hexdigest()
                                for f in ('inner_scores.csv','predictions.csv','selected_genes.csv')}
    safe_json(output,target/'report.json')
    print(f'FINAL_STATUS={output["status"]}; HELDOUT={heldout}; ELAPSED_SECONDS={output["elapsed_seconds"]}',flush=True)
    return target


Writing /content/cmps355_stage8/src/full_gene_loco.py


In [3]:
%%writefile /content/cmps355_stage8/src/stablemate_stage8_prepare.py
"""Stage 8: prepare leak-free, audit-ready StableMate LOCO inputs.

Uses the exactly preserved full-gene LASSO/Elastic-Net normalization and
train-only feature filter. This file does not fit or tune a model.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from full_gene_loco import (COHORTS, EXPECTED_ZIP_SHA, apply_features,
                            fit_features, load_data)


def sha(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(2**20), b''):
            h.update(chunk)
    return h.hexdigest()


def save_json_atomic(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(obj, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    os.replace(temporary, path)


def prepare(heldout: str, dataset_root: Path, work: Path, output: Path,
            r_source_hash: str, full_gene_hash: str, *, k: int = 2000,
            prefilt_p: int = 50, ncore: int = 2, bootstrap_b: int = 1000) -> dict:
    assert heldout in COHORTS
    meta, counts, genes, source = load_data(dataset_root)
    assert source['zip_sha256'] == EXPECTED_ZIP_SHA
    cohorts = meta['cohort'].to_numpy()
    train_idx = np.flatnonzero(cohorts != heldout)
    test_idx = np.flatnonzero(cohorts == heldout)
    assert len(train_idx) + len(test_idx) == 689
    assert not np.intersect1d(train_idx, test_idx).size
    y = (meta['class'] == 'CD').to_numpy(dtype=int)
    assert len(np.unique(y[train_idx])) == 2 and len(np.unique(y[test_idx])) == 2
    assert len(set(cohorts[train_idx])) == 3
    train_x, retained_idx, scaler, prep = fit_features(counts[train_idx])
    test_x = apply_features(counts[test_idx], retained_idx, scaler)
    assert np.isfinite(train_x).all() and np.isfinite(test_x).all()
    retained_genes = [genes[i] for i in retained_idx]
    assert len(retained_genes) == len(set(retained_genes))
    assert len(retained_genes) >= prefilt_p
    assert 'pseudo_predictor' not in retained_genes
    work.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    # Refuse a protocol/source change before touching any durable output file.
    prior_manifest = output / 'manifest.json'
    if prior_manifest.exists():
        prev = json.loads(prior_manifest.read_text(encoding='utf-8'))
        old_cfg = prev.get('model', {})
        old_code = prev.get('code_sha256', {})
        expected_code = {'r_runner': r_source_hash,
                         'full_gene_loco': full_gene_hash,
                         'stage8_prepare': sha(Path(__file__))}
        expected_controls = {'prefilter_p': prefilt_p, 'prefilter_K': k,
                             'model_K': k, 'bootstrap_B': bootstrap_b, 'ncore': ncore}
        if (prev.get('data_sha256') != EXPECTED_ZIP_SHA or
            prev.get('heldout') != heldout or old_code != expected_code or
            any(old_cfg.get(key) != value for key, value in expected_controls.items())):
            raise RuntimeError('Existing StableMate run has different data, settings or source; refusing to touch original outputs')
    for split, indices, matrix in [('train', train_idx, train_x), ('test', test_idx, test_x)]:
        frame = pd.DataFrame(matrix, columns=retained_genes)
        frame.insert(0, 'ID', meta.iloc[indices]['ID'].astype(str).to_numpy())
        frame.to_csv(work/f'{split}_x.csv', index=False, float_format='%.17g')
        clinical = meta.iloc[indices][['ID', 'cohort', 'Study', 'class']].copy()
        clinical['y'] = y[indices]
        clinical.to_csv(work/f'{split}_meta.csv', index=False)
    transform = pd.DataFrame({
        'gene': retained_genes,
        'train_mean_log2cpm': scaler.mean_,
        'train_std_log2cpm': scaler.scale_,
        'shared_universe_index': retained_idx,
    })
    transform_file = output/'gene_transform.csv'
    transform.to_csv(transform_file, index=False, float_format='%.17g')
    split_file = output/'outer_split.csv'
    meta[['ID','cohort','Study','class']].assign(
        split=np.where(cohorts == heldout, 'test', 'train')
    ).to_csv(split_file, index=False)
    manifest = {
        'method': 'StableMate',
        'track': 'adapted_author_method_loco_not_paper_reproduction',
        'heldout': heldout,
        'source_commit': 'cf0fe9f344c1756b6f89fd33f9fdae1a7f35b24a',
        'package_version_expected': '0.1.0',
        'dataset': source,
        'data_sha256': EXPECTED_ZIP_SHA,
        'training_cohorts': sorted(set(cohorts[train_idx].tolist())),
        'train_patients': int(len(train_idx)),
        'test_patients': int(len(test_idx)),
        'class_counts_train': {'CD': int(y[train_idx].sum()), 'Not IBD': int(len(train_idx) - y[train_idx].sum())},
        'class_counts_test': {'CD': int(y[test_idx].sum()), 'Not IBD': int(len(test_idx) - y[test_idx].sum())},
        'shared_genes': int(len(genes)),
        'eligible_genes': len(retained_genes),
        'preprocessing': {
            'method': 'exact full_gene_loco.fit_features and apply_features',
            'normalization': 'log2(1 + CPM) on fixed 9983-gene universe',
            'filter': 'CPM > 1 in >=10% training samples; nonzero training variance',
            'scaler': 'training-only sklearn StandardScaler',
            **prep,
        },
        'model': {
            'prefilter_p': prefilt_p,
            'prefilter_K': k,
            'model_K': k,
            'bootstrap_B': bootstrap_b,
            'ncore': ncore,
            'prefilter_family': 'binomial',
            'prefilter_group_by': 'Yenv',
            'st2_regressor': 'StableMate namespace reg_logit',
            'predictivity_objective': 'StableMate namespace obj_bic_logit',
            'stability_objective': 'StableMate namespace obj_pss_logit',
            'native_prediction': 'predict(...,pred_fun=StableMate:::pred_logit)',
            'paper_K_case_studies': 2000,
            'interpretation': 'K=2000 paper case-study ensemble size; this remains an adapted Crohn disease cohort benchmark, not a published-result replication',
            'seed': 2026 + int(heldout[-1]),
        },
        'code_sha256': {'r_runner': r_source_hash, 'full_gene_loco': full_gene_hash,
                        'stage8_prepare': sha(Path(__file__))},
        'environment': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__},
        'input_files_sha256': {p.name: sha(p) for p in sorted(work.glob('*.csv'))},
        'audit_artifact_sha256': {'gene_transform.csv':sha(transform_file),'outer_split.csv':sha(split_file)},
        'created_utc': datetime.now(timezone.utc).isoformat(),
    }
    manifest['run_key'] = hashlib.sha256(json.dumps({k:manifest[k] for k in
       ('method','heldout','source_commit','data_sha256','model','code_sha256')},
       sort_keys=True).encode()).hexdigest()
    existing=output/'manifest.json'
    if existing.exists():
        previous=json.loads(existing.read_text())
        if previous.get('run_key') != manifest['run_key']:
            raise RuntimeError('Existing output uses different model settings or sources; refusing to overwrite')
    save_json_atomic(existing, manifest)
    assert len(pd.read_csv(work/'train_x.csv',usecols=['ID'])) == len(train_idx)
    assert len(pd.read_csv(work/'test_x.csv',usecols=['ID'])) == len(test_idx)
    print(f'STAGE8_INPUT_PREPARED heldout={heldout} train={len(train_idx)} test={len(test_idx)} eligible_genes={len(retained_genes)}')
    return manifest


if __name__=='__main__':
    p=argparse.ArgumentParser()
    p.add_argument('--heldout',required=True,choices=COHORTS)
    p.add_argument('--data',type=Path,required=True)
    p.add_argument('--work',type=Path,required=True)
    p.add_argument('--out',type=Path,required=True)
    p.add_argument('--r-source-hash',required=True)
    p.add_argument('--full-gene-hash',required=True)
    p.add_argument('--K',type=int,default=2000)
    p.add_argument('--prefilt-p',type=int,default=50)
    p.add_argument('--cores',type=int,default=2)
    p.add_argument('--bootstrap',type=int,default=1000)
    a=p.parse_args()
    prepare(a.heldout,a.data,a.work,a.out,a.r_source_hash,a.full_gene_hash,
            k=a.K,prefilt_p=a.prefilt_p,ncore=a.cores,bootstrap_b=a.bootstrap)


Writing /content/cmps355_stage8/src/stablemate_stage8_prepare.py


In [4]:
%%writefile /content/cmps355_stage8/src/run_stablemate_stage8.R
#!/usr/bin/env Rscript
# Official StableMate LOCO adaptation. No reimplementation of its selector.
options(stringsAsFactors = FALSE)
args <- commandArgs(trailingOnly = TRUE)
if (length(args) != 3L) stop('Usage: Rscript run_stablemate_stage8.R INPUT_WORKDIR DRIVE_OUTPUT R_LIBRARY')
work <- normalizePath(args[[1L]], mustWork=TRUE)
out <- normalizePath(args[[2L]], mustWork=TRUE)
lib <- normalizePath(args[[3L]], mustWork=TRUE)
Sys.setenv(R_LIBS=lib,R_LIBS_USER=lib)
.libPaths(unique(c(lib,.libPaths())))
if (!requireNamespace('StableMate',quietly=TRUE)) stop('Official StableMate package not installed')
if (!requireNamespace('jsonlite',quietly=TRUE)) stop('jsonlite package missing')
if (!requireNamespace('foreach',quietly=TRUE) || !requireNamespace('doParallel',quietly=TRUE))
  stop('StableMate parallel dependencies missing')

expected_sha <- 'cf0fe9f344c1756b6f89fd33f9fdae1a7f35b24a'
metadata <- utils::packageDescription('StableMate')
actual_sha <- tolower(as.character(metadata[['RemoteSha']]))
if (!identical(actual_sha,expected_sha)) stop('Official source revision mismatch: ',actual_sha)
if (!identical(as.character(utils::packageVersion('StableMate')),'0.1.0')) stop('Official package version mismatch')
manifest <- jsonlite::read_json(file.path(out,'manifest.json'),simplifyVector=TRUE)
if (!identical(manifest$source_commit,expected_sha)) stop('Manifest source commit mismatch')
if (manifest$model$model_K != 2000L || manifest$model$prefilter_K != 2000L ||
    manifest$model$prefilter_p != 50L) stop('Unexpected K=2000 protocol settings')
if (manifest$model$ncore < 1L || manifest$model$ncore > 2L) stop('ncore must be 1 or 2')

ns <- asNamespace('StableMate')
required <- c('lasso_prefilt','stablemate','reg_logit','obj_bic_logit','obj_pss_logit','pred_logit')
missing <- required[!vapply(required,exists,logical(1),envir=ns,inherits=FALSE)]
if (length(missing)) stop('Missing required official functions: ',paste(missing,collapse=', '))
api <- file.path(out,'author_api_formals.txt')
con <- file(api,open='wt')
for (nm in required) {
  cat('FUNCTION=',nm,'\n',sep='',file=con)
  cat(paste(capture.output(print(formals(get(nm,envir=ns)))),collapse='\n'),'\n',file=con)
}
close(con)
cat('OFFICIAL_STABLEMATE_SHA=',actual_sha,'\n',sep='')
cat('OFFICIAL_STABLEMATE_VERSION=',as.character(utils::packageVersion('StableMate')),'\n',sep='')
cat('MANIFEST_RUN_KEY=',manifest$run_key,'\n',sep='')
cat('R_VERSION=',as.character(getRversion()),'\n',sep='')
cat('AUTHOR_API_VERIFIED=TRUE\n')

read_x <- function(split) {
  df <- utils::read.csv(file.path(work,paste0(split,'_x.csv')),check.names=FALSE,stringsAsFactors=FALSE)
  if (names(df)[1L] != 'ID' || anyDuplicated(names(df)) || anyNA(df$ID))
    stop('Invalid ',split,' feature matrix')
  X <- as.matrix(df[,-1L,drop=FALSE]); storage.mode(X) <- 'double'
  if (any(!is.finite(X)) || ncol(X) < 50L) stop('Non-finite/insufficient predictors: ',split)
  list(id=as.character(df$ID),X=X)
}
tr <- read_x('train');te <- read_x('test')
if (!identical(colnames(tr$X),colnames(te$X))) stop('Train/test gene mismatch')
if (anyDuplicated(c(tr$id,te$id))) stop('Patient ID overlap')
read_metadata <- function(split) {
  df <- utils::read.csv(file.path(work,paste0(split,'_meta.csv')),check.names=FALSE,stringsAsFactors=FALSE)
  if (!all(c('ID','class','y','cohort','Study') %in% names(df))) stop('Missing metadata fields')
  if (!identical(as.character(df$ID),if(split=='train')tr$id else te$id)) stop('Patient ID order mismatch')
  if (!all(df$y==as.integer(df$class=='CD'))) stop('Label encoding mismatch')
  df
}
mtrain <- read_metadata('train');mtest <- read_metadata('test')
ytrain <- as.integer(mtrain$y)
ytest <- as.integer(mtest$y)
env <- as.character(mtrain$cohort)
if (!identical(sort(unique(env)),sort(manifest$training_cohorts))) stop('Environment mismatch')
if (length(unique(env)) != 3L || length(unique(ytrain)) != 2L) stop('Invalid training cohort/label structure')
if (any(mtest$cohort %in% env) || any(mtest$cohort != manifest$heldout)) stop('Outer cohort leaked into training')
if (nrow(tr$X)!=manifest$train_patients || nrow(te$X)!=manifest$test_patients ||
    ncol(tr$X)!=manifest$eligible_genes) stop('Manifest dimensions mismatch')
set.seed(as.integer(manifest$model$seed))

save_atomic_rds <- function(value,path) {
  tmp <- paste0(path,'.tmp')
  saveRDS(value,tmp,compress=TRUE)
  if (!file.rename(tmp,path)) stop('Could not atomically publish RDS: ',path)
}
load_checkpoint <- function(path) {
  if (!file.exists(path)) return(NULL)
  tryCatch(readRDS(path),error=function(e)stop('Corrupted checkpoint: ',path,'; ',conditionMessage(e)))
}
check_phase_key <- function(phase) {
  key_file <- file.path(out,paste0(phase,'.run_key'))
  if (file.exists(key_file) && !identical(readLines(key_file,warn=FALSE),manifest$run_key))
    stop('Existing ',phase,' checkpoint is from a different run')
  key_file
}

pool_file <- file.path(out,'prefilter_pool.rds')
pool_key <- check_phase_key('prefilter')
if (file.exists(pool_file)) {
  if (!file.exists(pool_key)) stop('Prefilter checkpoint has no matching run key; refuse unsafe reuse')
  pool <- load_checkpoint(pool_file)
  cat('PHASE_PREFILTER=RESTORED\n')
} else {
  cat('PHASE_PREFILTER=START\n'); flush.console()
  # Independent phase seed makes resume from a completed prefilter checkpoint consistent.
  set.seed(as.integer(manifest$model$seed) + 101L)
  pool <- StableMate::lasso_prefilt(Y=ytrain,X=tr$X,env=env,
    p=as.integer(manifest$model$prefilter_p),K=as.integer(manifest$model$prefilter_K),
    ncore=as.integer(manifest$model$ncore),par_method='MC',family='binomial',
    group_by='Yenv',verbose=FALSE)
  if (length(pool)==0L) stop('Official prefilter returned empty predictor pool')
  writeLines(manifest$run_key,pool_key)
  save_atomic_rds(pool,pool_file)
  cat('PHASE_PREFILTER=SAVED\n')
}
writeLines(c(paste('class:',paste(class(pool),collapse=',')),
             paste('dimensions:',paste(dim(pool),collapse=' x ')),
             capture.output(str(pool,max.level=1,give.attr=FALSE))),
           file.path(out,'prefilter_pool_structure.txt'))

fit_file <- file.path(out,'official_model.rds')
fit_key <- check_phase_key('model')
if (file.exists(fit_file)) {
  if (!file.exists(fit_key)) stop('Model checkpoint has no matching run key; refuse unsafe reuse')
  model <- load_checkpoint(fit_file)
  cat('PHASE_OFFICIAL_MODEL=RESTORED\n')
} else {
  cat('PHASE_OFFICIAL_MODEL=START\n');flush.console()
  # Do not depend on RNG consumed by the (possibly restored) prefilter phase.
  set.seed(as.integer(manifest$model$seed) + 202L)
  model <- StableMate::stablemate(Y=ytrain,X=tr$X,env=env,
    K=as.integer(manifest$model$model_K),
    ncore=as.integer(manifest$model$ncore),par_method='MC',
    pred_st2_ctrl=list(reg_fun=get('reg_logit',ns),obj_fun=get('obj_bic_logit',ns),pred_pool=pool),
    stab_st2_ctrl=list(reg_fun=get('reg_logit',ns),obj_fun=get('obj_pss_logit',ns)),
    pred_calc_imp_ctrl=list(B=as.integer(manifest$model$bootstrap_B)),
    verbose=FALSE)
  if (is.null(model) || !length(model)) stop('Official model returned empty result')
  writeLines(manifest$run_key,fit_key)
  save_atomic_rds(model,fit_file)
  cat('PHASE_OFFICIAL_MODEL=SAVED\n')
}

# Author model prediction; do not silently substitute a new classifier.
prob <- as.numeric(stats::predict(model,X=te$X,pred_fun=get('pred_logit',ns)))
if (length(prob)!=length(ytest) || any(!is.finite(prob)) || any(prob<0 | prob>1))
  stop('Official model did not return one valid probability per test patient')
pred <- data.frame(outer=manifest$heldout,method='stablemate_author_native',
  ID=as.character(mtest$ID),Study=as.character(mtest$Study),
  class_true=as.character(mtest$class),y_true=ytest,
  probability_CD=prob,y_pred=as.integer(prob>=0.5))
write.csv(pred,file.path(out,'native_predictions.csv'),row.names=FALSE)

# Preserve native scores and selection definitions. A zero-selected result is valid.
if (is.null(model$prediction_ensemble$imp_scores$joint) ||
    is.null(model$stable_ensemble$imp_scores$conditional))
  stop('Author output shape changed; original RDS remains saved for forensic review')
ps <- model$prediction_ensemble$imp_scores$joint
ss <- model$stable_ensemble$imp_scores$conditional
save_atomic_rds(list(prediction=ps,stability=ss),file.path(out,'official_importance_objects.rds'))
if (is.null(ps$importance) || is.null(ps$significance) ||
    is.null(ss$importance) || is.null(ss$significance))
  stop('Official score object missing importance/significance; no invented scores')
genes <- intersect(names(ps$importance),names(ss$importance))
genes <- setdiff(genes,'pseudo_predictor')
if (!length(genes) || anyDuplicated(genes)) stop('No correctly named author gene scores')
scores <- data.frame(gene=genes,
    predictive_importance=as.numeric(ps$importance[genes]),
    stability_importance=as.numeric(ss$importance[genes]),
    predictive_significance=as.numeric(ps$significance[genes]),
    stability_significance=as.numeric(ss$significance[genes]))
scores$significantly_predictive <- !is.na(scores$predictive_significance) &
  scores$predictive_significance > 0.975
scores$significantly_stable_and_predictive <- scores$significantly_predictive &
  !is.na(scores$stability_significance) & scores$stability_significance > 0.975
scores <- scores[order(-scores$predictive_importance,scores$gene),]
write.csv(scores,file.path(out,'official_gene_scores.csv'),row.names=FALSE)
selected <- scores[scores$significantly_stable_and_predictive,,drop=FALSE]
write.csv(selected,file.path(out,'significant_stable_genes.csv'),row.names=FALSE)
write.csv(head(scores,50L),file.path(out,'diagnostic_top50_predictivity.csv'),row.names=FALSE)
cat('OFFICIAL_SCORED_GENES=',nrow(scores),'\n',sep='')
cat('OFFICIAL_SIGNIFICANT_PREDICTIVE_GENES=',sum(scores$significantly_predictive),'\n',sep='')
cat('OFFICIAL_SIGNIFICANT_STABLE_GENES=',nrow(selected),'\n',sep='')
cat('NATIVE_HELDOUT_PREDICTIONS=',nrow(pred),'\n',sep='')
cat('R_SESSION_BEGIN\n');print(sessionInfo());cat('R_SESSION_END\n')
cat('STAGE8_R_COMPLETE=TRUE\n')


Writing /content/cmps355_stage8/src/run_stablemate_stage8.R


## Pinned R environment

In [5]:
# Restore the previously verified R-library cache (if available).
if shutil.which('Rscript') is None:
    subprocess.run(['bash','-lc','apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq r-base'], check=True)
R = shutil.which('Rscript')
assert R
rver = subprocess.check_output([R,'--vanilla','-e','cat(as.character(getRversion()))'],text=True).strip()
osinfo=platform.freedesktop_os_release()
ck=f"r{rver}_{osinfo.get('ID','linux')}{osinfo.get('VERSION_ID','unknown')}_{platform.machine()}"
cache_root=ROOT/'runs/environment_cache'
cache_file=cache_root/f'{ck}.tar.gz'
cache_manifest=cache_root/f'{ck}.json'

def filehash(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

def installed_sha():
    exp="lib<-Sys.getenv('CMPS355_R_LIB');.libPaths(c(lib,.libPaths()));if(!requireNamespace('StableMate',quietly=TRUE))quit(status=4L);cat(utils::packageDescription('StableMate')[['RemoteSha']])"
    env=os.environ.copy();env['CMPS355_R_LIB']=str(LIB)
    x=subprocess.run([R,'--vanilla','-e',exp],text=True,capture_output=True,env=env)
    return x.stdout.strip().lower() if x.returncode==0 else None

if installed_sha()!=SHA and cache_file.is_file() and cache_manifest.is_file():
    cm=json.loads(cache_manifest.read_text())
    assert cm['sha256']==filehash(cache_file),'R library cache checksum mismatch'
    assert cm['r_version']==rver and cm['os']==osinfo.get('ID','linux') and cm['os_version']==osinfo.get('VERSION_ID','unknown') and cm['arch']==platform.machine(),'R cache is incompatible with current runtime'
    with tempfile.TemporaryDirectory(dir='/content') as tmp:
        with tarfile.open(cache_file,'r:gz') as z:
            assert all(x.name.startswith('r_library/') and '..' not in Path(x.name).parts for x in z.getmembers()),'Unsafe cache layout'
            z.extractall(tmp,filter='data')
        for d in (Path(tmp)/'r_library').iterdir():
            if d.is_dir() and not (LIB/d.name).exists():
                shutil.copytree(d,LIB/d.name)
    print('Restored pinned package library from:',cache_file)

if installed_sha()!=SHA:
    # Fallback: install the author package at the fixed commit. Not an approximation.
    subprocess.run(['bash','-lc', 'apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq r-cran-remotes r-cran-glmnet r-cran-foreach r-cran-doparallel r-cran-arm r-cran-dplyr r-cran-ggplot2 r-cran-reshape2 r-cran-progress r-cran-jsonlite'],check=True)
    exp=f".libPaths(c('{LIB}',.libPaths()));remotes::install_github('meiosis97/StableMate',ref='{SHA}',subdir='pkg',lib='{LIB}',dependencies=NA,upgrade='never',force=TRUE,build_vignettes=FALSE)"
    env=os.environ.copy();env['CMPS355_R_LIB']=str(LIB)
    install_log=OUT/'official_package_install.log'
    with install_log.open('w') as log:
        result=subprocess.run([R,'--vanilla','-e',exp],stdout=log,stderr=subprocess.STDOUT,env=env)
    assert result.returncode==0,f'Official package installation failed; see {install_log}'
    assert installed_sha()==SHA,'Installation installed the wrong official source revision'
    print('Pinned official R package installed. Log:',install_log)

assert installed_sha()==SHA,'Official StableMate revision could not be verified'
print('Official StableMate revision:', SHA, 'R:',rver)


Restored pinned package library from: /content/drive/MyDrive/CMPS355_Research/runs/environment_cache/r4.6.1_ubuntu24.04_x86_64.tar.gz
Pinned official R package installed. Log: /content/drive/MyDrive/CMPS355_Research/runs/stablemate_k2000/d3/official_package_install.log
Official StableMate revision: cf0fe9f344c1756b6f89fd33f9fdae1a7f35b24a R: 4.6.1


## Validated cohorts and train-only expression preprocessing

In [6]:
# Prepare inputs using exactly the same train-only log-CPM and scaling code as Stage 3.
prep_code=SRC/'stablemate_stage8_prepare.py'
r_code=SRC/'run_stablemate_stage8.R'
base_code=SRC/'full_gene_loco.py'
subprocess.run([sys.executable,str(prep_code),'--heldout',HELDOUT,'--data',str(DATA.parent),
                '--work',str(WORK),'--out',str(OUT),
                '--r-source-hash',filehash(r_code),'--full-gene-hash',filehash(base_code),
                '--K',str(K),'--prefilt-p',str(PREFILTER_P),
                '--cores',str(NCORE),'--bootstrap',str(BOOTSTRAP_B)],check=True)
manifest=json.loads((OUT/'manifest.json').read_text())
assert manifest['data_sha256']=='8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74'
print('Run key:',manifest['run_key'])
print('Input train/test:',manifest['train_patients'],manifest['test_patients'])
print('Eligible genes:',manifest['eligible_genes'])


Run key: 5d2d2a97c9052d91f4dae98ce0e76b411f6b602de7def3e4b3ebdc9107ae7847
Input train/test: 613 76
Eligible genes: 8409


## Official StableMate author training and prediction

In [7]:
# Execute only the official R package. Checkpoints are saved after prefilter and fit.
import signal
report=OUT/'report.json'
if report.exists() and json.loads(report.read_text()).get('status','').startswith('PASS'):
    print('Completed run already verified:',report)
else:
    env=os.environ.copy()
    env['CMPS355_R_LIB']=str(LIB)
    cmd=[R,'--vanilla',str(SRC/'run_stablemate_stage8.R'),str(WORK),str(OUT),str(LIB)]
    runlog=OUT/'official_run.log'
    with runlog.open('a',encoding='utf-8') as log:
        log.write('\n===== EXECUTION '+datetime.now(timezone.utc).isoformat()+' =====\n')
        log.flush()
        proc=subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,env=env)
        begun=time.monotonic()
        while proc.poll() is None:
            time.sleep(60)
            elapsed=round((time.monotonic()-begun)/60,1)
            checkpoints=[name for name in ('prefilter_pool.rds','official_model.rds','native_predictions.csv') if (OUT/name).is_file()]
            if int(elapsed) % 5 == 0:
                print(f'elapsed={elapsed} min; checkpoints={checkpoints}; log={runlog}',flush=True)
        code=proc.wait()
    tail=runlog.read_text(errors='replace').splitlines()[-30:]
    print('\n'.join(tail))
    assert code==0,f'StableMate process exited with {code}; inspect {runlog}. Rerun notebook to resume completed phases.'


elapsed=5.0 min; checkpoints=[]; log=/content/drive/MyDrive/CMPS355_Research/runs/stablemate_k2000/d3/official_run.log
elapsed=10.0 min; checkpoints=[]; log=/content/drive/MyDrive/CMPS355_Research/runs/stablemate_k2000/d3/official_run.log
elapsed=15.0 min; checkpoints=[]; log=/content/drive/MyDrive/CMPS355_Research/runs/stablemate_k2000/d3/official_run.log
elapsed=20.0 min; checkpoints=['prefilter_pool.rds']; log=/content/drive/MyDrive/CMPS355_Research/runs/stablemate_k2000/d3/official_run.log


KeyboardInterrupt: 

## Verification and report

In [ ]:
# Independent verification of native held-out predictions and complete artifact inventory.
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, f1_score, confusion_matrix
import pandas as pd, numpy as np
from pathlib import Path
pred=OUT/'native_predictions.csv'
assert pred.is_file(),f'Missing native predictions: {pred}'
p=pd.read_csv(pred,dtype={'ID':str,'Study':str})
assert len(p)==manifest['test_patients']
assert p['ID'].is_unique and set(p['ID'])==set(pd.read_csv(WORK/'test_meta.csv',dtype={'ID':str})['ID'])
assert (p['method']=='stablemate_author_native').all()
assert set(p['y_true'].unique())=={0,1}
assert np.isfinite(p['probability_CD']).all() and p['probability_CD'].between(0,1).all()
meta_test=pd.read_csv(WORK/'test_meta.csv',dtype={'ID':str})
validated=p.set_index('ID').loc[meta_test['ID']]
assert (validated['y_true'].to_numpy()==meta_test['y'].to_numpy()).all(), 'Test labels mismatched'
score_file=OUT/'official_gene_scores.csv'
select_file=OUT/'significant_stable_genes.csv'
assert score_file.is_file() and select_file.is_file()
scores=pd.read_csv(score_file)
selected=pd.read_csv(select_file)
assert len(scores)>0 and len(selected)==scores['significantly_stable_and_predictive'].sum()
assert set(selected['gene']).issubset(set(scores['gene']))
y=p['y_true'].to_numpy(); pr=p['probability_CD'].to_numpy(); hard=(pr>=.5).astype(int)
tn,fp,fn,tp=[int(z) for z in confusion_matrix(y,hard,labels=[0,1]).ravel()]
metrics={'auroc':float(roc_auc_score(y,pr)), 'average_precision':float(average_precision_score(y,pr)),
         'balanced_accuracy':float(balanced_accuracy_score(y,hard)),
         'f1':float(f1_score(y,hard,zero_division=0)),'sensitivity':tp/max(tp+fn,1),
         'specificity':tn/max(tn+fp,1),'tp':tp,'tn':tn,'fp':fp,'fn':fn}
files=sorted(x for x in OUT.iterdir() if x.is_file() and x.name not in {'report.json','progress.json'})
report_obj={'status':'PASS_NATIVE_WITH_STABLE_GENES' if len(selected)>0 else 'PASS_NATIVE_NO_SIGNIFICANT_STABLE_GENES',
            'method':'StableMate','analysis_track':'author_package_adapted_loco_not_published_replication',
            'heldout':HELDOUT,'dataset_sha256':manifest['data_sha256'],
            'author_commit':SHA,'package_version':'0.1.0','model_K':K,'paper_case_study_K':2000,
            'training_environment_cohorts':manifest['training_cohorts'],
            'n_train':manifest['train_patients'],'n_test':manifest['test_patients'],
            'eligible_genes':manifest['eligible_genes'],'author_scored_genes':int(len(scores)),
            'significant_predictive_genes':int(scores['significantly_predictive'].sum()),
            'significant_stable_genes':int(len(selected)),
            'native_heldout_metrics':metrics,
            'interpretation_note':'These are the native StableMate ensemble probabilities, not a common downstream classifier. K=2000 matches the paper case-study ensemble size; other adaptations remain.',
            'comparison_note':'StableMate requires multiple training environments; single-cohort independent fit is not equivalent and is not supplied.',
            'reproducibility':{'manifest_run_key':manifest['run_key'],'r_version':rver,
                'input_file_hashes':manifest['input_files_sha256'],
                'artifact_sha256':{x.name:filehash(x) for x in files}},
            'completed_utc':datetime.now(timezone.utc).isoformat()}
report_tmp=OUT/'report.json.tmp'
report_tmp.write_text(json.dumps(report_obj,indent=2,sort_keys=True)+'\n')
report_tmp.replace(OUT/'report.json')
print('FINAL_STAGE8_STATUS:',report_obj['status'])
print('Metrics:',metrics)
print('Significant StableMate genes:',len(selected))
print('All outputs:',OUT)
